In [1]:
import feedparser
import pandas as pd
import re
import time

from datetime import datetime
from urllib.parse import quote_plus

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
"""
=========================================================
EVENT DRIVEN NEWS MONITOR
=========================================================

Architecture

EVENT BUCKET QUERIES
    ↓
RSS ENGINES (Google India + Google Global + Bing)
    ↓
RAW ARTICLE COLLECTION
    ↓
SAVE RAW FILE
    ↓
COMPANY ALIAS DETECTION (Regex)
EVENT KEYWORD DETECTION (Regex)
    ↓
STRUCTURED COMPANY NEWS
    ↓
SAVE PROCESSED FILE

Author: Kaustubh
Purpose: Adverse news monitoring for companies
=========================================================
"""

import feedparser
import pandas as pd
import re
import time

from datetime import datetime, timedelta
from urllib.parse import quote_plus


# =========================================================
# CONFIGURATION
# =========================================================

# RSS engines used for ingestion
GOOGLE_INDIA = "https://news.google.com/rss/search?q={query}&hl=en-IN&gl=IN&ceid=IN:en"
GOOGLE_GLOBAL = "https://news.google.com/rss/search?q={query}&hl=en&gl=US&ceid=US:en"
BING = "https://www.bing.com/news/search?q={query}&format=rss"

RSS_ENGINES = {
    "google_india": GOOGLE_INDIA,
    "google_global": GOOGLE_GLOBAL,
    "bing": BING
}

# delay between requests (polite scraping)
FETCH_DELAY = 1

# how far back to consider articles
TIME_WINDOW_HOURS = 30


# =========================================================
# EVENT BUCKETS
# =========================================================
# Logical grouping of adverse events

EVENT_BUCKETS = {

    "governance": [
        "fraud","lawsuit","investigation","probe","penalty",
        "violation","settlement","arbitration","cartel"
    ],

    "accident": [
        "fire","explosion","accident","injured","death","fatality"
    ],

    "environment": [
        "oil spill","toxic emission","water contamination",
        "deforestation","hazardous waste"
    ],

    "cyber": [
        "cyber attack","data breach","hack","cyber fraud"
    ],

    "regulatory": [
        "SEBI","RBI","FIR","money laundering","recall"
    ]

}


# =========================================================
# COMPANY ALIAS DICTIONARY
# =========================================================
# Each company can have multiple aliases

COMPANY_ALIASES = {

    "Tata Steel": ["tata steel","tata"],

    "Reliance Industries": [
        "reliance industries",
        "reliance"
    ],

    "Tata Consultancy Services": [
        "tcs",
        "tata consultancy services"
    ]

}


# =========================================================
# BUILD COMPANY REGEX
# =========================================================

alias_lookup = {}

for company, aliases in COMPANY_ALIASES.items():

    for alias in aliases:
        alias_lookup[alias.lower()] = company


company_pattern = re.compile(
    r"\b(" + "|".join(map(re.escape, alias_lookup.keys())) + r")\b",
    re.IGNORECASE
)


# =========================================================
# BUILD EVENT REGEX
# =========================================================

event_keywords = []

for bucket in EVENT_BUCKETS.values():
    event_keywords.extend(bucket)


event_pattern = re.compile(
    r"\b(" + "|".join(map(re.escape, event_keywords)) + r")\b",
    re.IGNORECASE
)


# =========================================================
# HELPER FUNCTIONS
# =========================================================

def detect_companies(text):
    """
    Detect company names in a text using regex.

    Returns list of companies found.
    """

    matches = company_pattern.findall(text)

    companies = set()

    for m in matches:
        companies.add(alias_lookup[m.lower()])

    return list(companies)


def detect_events(text):
    """
    Detect adverse event keywords in text.
    """

    return list(set(event_pattern.findall(text)))


# =========================================================
# RSS FETCH FUNCTION
# =========================================================

def fetch_bucket(bucket_name, keywords):
    """
    Fetch RSS results for a keyword bucket.
    """

    rows = []

    # build OR query
    query = " OR ".join([f'"{k}"' for k in keywords])

    encoded_query = quote_plus(query)

    for engine, url_template in RSS_ENGINES.items():

        url = url_template.format(query=encoded_query)

        feed = feedparser.parse(url)

        for entry in feed.entries:

            headline = entry.title
            link = entry.link

            # parse published date if available
            published = None

            if hasattr(entry, "published_parsed"):
                published = datetime(*entry.published_parsed[:6])

            rows.append({

                "bucket": bucket_name,
                "engine": engine,
                "headline": headline,
                "link": link,
                "published": published

            })

    return rows


# =========================================================
# MAIN FETCH PIPELINE
# =========================================================

def fetch_all_articles():

    all_rows = []

    for bucket, keywords in EVENT_BUCKETS.items():

        print(f"Fetching bucket: {bucket}")

        rows = fetch_bucket(bucket, keywords)

        all_rows.extend(rows)

        time.sleep(FETCH_DELAY)

    df = pd.DataFrame(all_rows)

    return df


# =========================================================
# PROCESS RAW DATA
# =========================================================

def process_articles(df):

    records = []

    for _, row in df.iterrows():

        text = row["headline"]

        companies = detect_companies(text)

        events = detect_events(text)

        if not companies:
            continue

        for company in companies:

            records.append({

                "company": company,
                "events": ",".join(events),
                "headline": row["headline"],
                "engine": row["engine"],
                "link": row["link"],
                "published": row["published"]

            })

    return pd.DataFrame(records)


# =========================================================
# MAIN PROGRAM
# =========================================================

def main():

    print("\nStarting event-driven news monitor...\n")

    # -----------------------------------------------------
    # STEP 1 - FETCH ARTICLES
    # -----------------------------------------------------

    raw_df = fetch_all_articles()

    print("Total articles fetched:", len(raw_df))


    # -----------------------------------------------------
    # STEP 2 - FILTER TIME WINDOW
    # -----------------------------------------------------

    cutoff = datetime.utcnow() - timedelta(hours=TIME_WINDOW_HOURS)

    raw_df = raw_df[raw_df["published"] > cutoff]


    # -----------------------------------------------------
    # STEP 3 - DEDUPLICATE BY URL
    # -----------------------------------------------------

    raw_df = raw_df.drop_duplicates(subset=["link"])


    # -----------------------------------------------------
    # STEP 4 - SAVE RAW DATA
    # -----------------------------------------------------

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")

    raw_path = f"raw_news_{timestamp}.csv"

    raw_df.to_csv(raw_path, index=False)

    print("Raw dataset saved:", raw_path)


    # -----------------------------------------------------
    # STEP 5 - ENTITY DETECTION
    # -----------------------------------------------------

    processed_df = process_articles(raw_df)


    # -----------------------------------------------------
    # STEP 6 - SORT RESULTS
    # -----------------------------------------------------

    processed_df = processed_df.sort_values(
        by=["company", "published"],
        ascending=[True, False]
    )


    # -----------------------------------------------------
    # STEP 7 - SAVE PROCESSED FILE
    # -----------------------------------------------------

    processed_path = f"company_news_{timestamp}.csv"

    processed_df.to_csv(processed_path, index=False)

    print("Processed dataset saved:", processed_path)

    print("\nRecords found:", len(processed_df))


# =========================================================
# ENTRY POINT
# =========================================================

if __name__ == "__main__":
    main()